In [1]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [2]:
PIPELINE_ROOT = Path("/mnt/primary/Finnhub Pipeline")
ANSWER_ROOT = PIPELINE_ROOT / "finnhub_answers"

START_DATE = pd.Timestamp("2026-07-15")
END_DATE = pd.Timestamp("2026-08-28")

OUTPUT_ROOT = PIPELINE_ROOT / "answer_change_results" / "tfidf"
GRAPH_DIR = OUTPUT_ROOT / "daily_graphs"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("Answer folder:", ANSWER_ROOT)
print("Period:", START_DATE.date(), "to", END_DATE.date())


Answer folder: /mnt/primary/Finnhub Pipeline/finnhub_answers
Period: 2026-07-15 to 2026-08-28


In [3]:
def extract_date(path: Path):
    for part in reversed(path.parts):
        try:
            date = pd.Timestamp(part).normalize()
            if START_DATE <= date <= END_DATE:
                return date
        except Exception:
            pass

    match = re.search(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if match:
        date = pd.Timestamp(match.group(1)).normalize()
        if START_DATE <= date <= END_DATE:
            return date

    return None


def extract_company(path: Path) -> str:
    name = re.sub(r"_answer_\d{4}-\d{2}-\d{2}$", "", path.stem, flags=re.IGNORECASE)
    name = re.sub(r"_answer$", "", name, flags=re.IGNORECASE)
    return name.strip()


def read_answer(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="replace")
    return re.sub(r"\s+", " ", text).strip()


if not ANSWER_ROOT.exists():
    raise FileNotFoundError(f"Answer folder not found: {ANSWER_ROOT}")

records = []

for path in ANSWER_ROOT.rglob("*.txt"):
    date = extract_date(path)

    if date is None:
        continue

    text = read_answer(path)

    if not text:
        continue

    records.append(
        {
            "Date": date,
            "Company": extract_company(path),
            "AnswerText": text,
        }
    )

answers = (
    pd.DataFrame(records)
    .drop_duplicates(subset=["Date", "Company"], keep="last")
    .sort_values(["Company", "Date"])
    .reset_index(drop=True)
)

if answers.empty:
    raise ValueError("No answer files were found in the selected period.")

print("Answers loaded:", len(answers))
print("Companies:", answers["Company"].nunique())
print("Date range:", answers["Date"].min().date(), "to", answers["Date"].max().date())


Answers loaded: 4400
Companies: 100
Date range: 2026-07-15 to 2026-08-28


In [4]:
change_rows = []

for company, group in answers.groupby("Company"):
    group = group.sort_values("Date").reset_index(drop=True)

    if len(group) < 2:
        continue

    vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
    matrix = vectorizer.fit_transform(group["AnswerText"])

    for i in range(1, len(group)):
        similarity = float(cosine_similarity(matrix[i], matrix[i - 1])[0, 0])
        change_rows.append(
            {
                "Company": company,
                "Date": group.loc[i, "Date"],
                "PreviousDate": group.loc[i - 1, "Date"],
                "TFIDF_Change": 1.0 - similarity,
            }
        )

daily_change = pd.DataFrame(change_rows)

if daily_change.empty:
    raise ValueError("No consecutive answer comparisons could be calculated.")

daily_change = daily_change.sort_values(["Company", "Date"]).reset_index(drop=True)


In [5]:
for company, group in daily_change.groupby("Company"):
    group = group.sort_values("Date").copy()

    peak_row = group.loc[group["TFIDF_Change"].idxmax()]
    peak_date = pd.Timestamp(peak_row["Date"])
    peak_change = float(peak_row["TFIDF_Change"])

    figure, axis = plt.subplots(figsize=(11, 5))
    axis.plot(group["Date"], group["TFIDF_Change"], marker="o")

    axis.set_xlim(START_DATE, END_DATE)
    axis.set_ylim(0.0, 1.0)
    axis.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

    axis.set_xlabel("Date")
    axis.set_ylabel("TF-IDF Change")
    axis.set_title(f"{company} — Daily Answer Change (TF-IDF)")
    axis.tick_params(axis="x", rotation=45)

    axis.annotate(
        f"Highest spike: {peak_date.strftime('%d %b %Y')}",
        xy=(peak_date, min(peak_change, 1.0)),
        xytext=(10, -25),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->"},
    )

    plt.tight_layout()

    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", company).strip("_")
    figure.savefig(GRAPH_DIR / f"{safe_name}_tfidf_daily_change.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

print("Saved daily graphs:", daily_change["Company"].nunique())
print("Graph folder:", GRAPH_DIR)


Saved daily graphs: 100
Graph folder: /mnt/primary/Finnhub Pipeline/answer_change_results/tfidf/daily_graphs


In [6]:
mean_change = (
    daily_change.groupby("Company", as_index=False)["TFIDF_Change"]
    .mean()
    .rename(columns={"TFIDF_Change": "Mean_TFIDF_Change"})
)

peak_rows = (
    daily_change.loc[daily_change.groupby("Company")["TFIDF_Change"].idxmax(), ["Company", "Date"]]
    .rename(columns={"Date": "Highest_Spike_Date"})
)

final_table = (
    mean_change.merge(peak_rows, on="Company", how="left")
    .sort_values("Mean_TFIDF_Change", ascending=False)
    .reset_index(drop=True)
)

final_table["Highest_Spike_Date"] = pd.to_datetime(final_table["Highest_Spike_Date"]).dt.strftime("%Y-%m-%d")
final_table.index = final_table.index + 1
final_table.index.name = "Rank"

display(final_table.round(4))


,Company,Mean_TFIDF_Change,Highest_Spike_Date
Rank,,,
1,Goldman Sachs,0.1303,2026-07-19
2,JPMorgan Chase,0.1252,2026-07-19
3,CVS Health,0.1150,2026-07-23
4,AT&T,0.1150,2026-07-16
5,Xcel Energy,0.1132,2026-07-23
6,Boeing,0.1120,2026-07-22
7,Caterpillar Inc,0.1103,2026-07-22
8,"Tesla, Inc",0.1097,2026-07-17
9,Accenture,0.1075,2026-07-19


In [7]:
daily_change.to_csv(OUTPUT_ROOT / "tfidf_daily_change.csv", index=False)
final_table.to_csv(OUTPUT_ROOT / "tfidf_company_ranking.csv", index=True)

print("Saved results to:", OUTPUT_ROOT)


Saved results to: /mnt/primary/Finnhub Pipeline/answer_change_results/tfidf
